# **LAB DL Session 4: Word Embedding Techniques** (MATERI KUIS SESI 4-5, untuk kuis di sesi 6)
Translasi kata ke dalam vektor numerik yang dapat digunakan oleh model machine learning.
untuk prediksi teks2 berikutnya

Step 1: Tokenization
- Data Raw -> Tokenization -> List of Tokens
- Standarisasi adalah proses mengubah teks menjadi bentuk yang lebih konsisten, misalnya mengubah semua huruf menjadi huruf kecil, menghapus tanda baca, dan menghapus karakter khusus.
- Contoh: "Saya suka makan nasi goreng." -> "saya", "suka", "makan", "nasi", "goreng"
- Padding 0 = kasih tau gada isi

Step 2: Stop Words Removal
- Stop Words adalah kata-kata umum yang sering muncul dalam teks tetapi tidak memberikan banyak informasi, seperti "dan", "atau", "adalah", dll. Proses ini melibatkan menghapus kata-kata tersebut dari daftar token.
- List of Tokens -> Stop Words Removal -> Cleaned Tokens

RNN : Read Word by Word
RNN juga punya masalah, ex di ai agent suka lupa context.

RNN maksa untuk baca dataset yang besar, tapi RNN punya masalah dengan long-term dependencies. Jadi kalau ada kata yang penting di awal, RNN bisa lupa karena terlalu banyak informasi yang harus diingat.

Maka, pake LSTM (Long Short-Term Memory) atau GRU (Gated Recurrent Unit) untuk mengatasi masalah ini. LSTM dan GRU memiliki mekanisme untuk mengingat informasi penting dalam jangka panjang, sehingga lebih efektif dalam menangani konteks yang panjang.


In [8]:
# Uncomment jika pakai google colab karena Google Colab tidak memiliki semua library yang dibutuhkan.
from google.colab import files
import zipfile
import os

uploaded = files.upload()

print("Selesai! Struktur dataset:")

Saving ind.txt to ind.txt
Selesai! Struktur dataset:


In [9]:
# Import library

import tensorflow as tf
import re
import numpy as np
from sklearn.model_selection import train_test_split

In [10]:
dataset = tf.data.TextLineDataset("/content/ind.txt")

In [11]:
VOCAB_SIZE = 2000 # 2000 Token ID dari 2000 kata-kata yang paling sering muncul
ENG_SEQ_LENGTH = 16 # padding 16 slots untuk kalimat bahasa inggris
IND_SEQ_LENGTH = 16 # padding 16 slots untuk kalimat bahasa indonesia
EMBEDDING_DIM = 256
BATCH_SIZE = 32 # berapa banyak kalimat yang diproses dalam satu batch
HIDDEN_UNITS = 256 # untuk memory lstm

# **Standarisasi Data**

- Text Vectorization untuk keduanya (english dan indonesia) agar bisa diinput ke model.
- -> standarisasi ("", .?, dll) supaya semua kata sama dan clean.

In [12]:
eng_vec_layer = tf.keras.layers.TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_sequence_length=ENG_SEQ_LENGTH,
)

ind_vec_layer = tf.keras.layers.TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_sequence_length=IND_SEQ_LENGTH,
)

In [28]:
def preprocess(text):
    text = tf.strings.split(text, "\t")
    eng = text[0]
    ind = "start> " + text[1] + "<end>"
    return eng, ind

def vectorize(text):
    text = tf.strings.split(text, "\t")
    eng_sentence = text[0]
    ind_raw_sentence = text[1]

    decoder_input = "start> " + ind_raw_sentence
    decoder_target = ind_raw_sentence + "<end>"

    return{
        "input_1": eng_vec_layer(eng_sentence),
        "input_2": ind_vec_layer(decoder_input)
    }, ind_vec_layer(decoder_target)

In [ ]:
# # ini kalo di lab benernya

# def preprocess(text):
#     text = tf.strings.split(text, "\t")
#     eng = text[0]
#     ind = "start> " + text[1] + "<end>"
#     return eng, ind

# def vectorize(text):
#     text = tf.strings.split(text, "\t")
#     eng = text[0]
#     start = "<starts>" + text[1]
#     end = text[1] + "<end>"

#     return{
#         "input_1": eng_vec_layer(end),
#         "input_2": ind_vec_layer(ind)
#     }, ind_vec_layer(end)

In [22]:
preprocessed_dataset = dataset.map(preprocess)

# hitung berapa kali kata yang muncul di dataset untuk membuat kamus kata (vocabulary)
eng_adapt_dataset = preprocessed_dataset.map(lambda eng, ind: eng)
eng_vec_layer.adapt(eng_adapt_dataset)

ind_adapt_dataset = preprocessed_dataset.map(lambda eng, ind: ind)
ind_vec_layer.adapt(ind_adapt_dataset)

# lihat hasil
index_to_word = {x: y for x, y in zip(range(len(ind_vec_layer.get_vocabulary())), ind_vec_layer.get_vocabulary())}

print(index_to_word)

{0: '', 1: '[UNK]', 2: np.str_('start'), 3: np.str_('aku'), 4: np.str_('tom'), 5: np.str_('tidak'), 6: np.str_('yang'), 7: np.str_('di'), 8: np.str_('kamu'), 9: np.str_('saya'), 10: np.str_('dia'), 11: np.str_('itu'), 12: np.str_('ini'), 13: np.str_('akan'), 14: np.str_('apa'), 15: np.str_('ada'), 16: np.str_('bisa'), 17: np.str_('dengan'), 18: np.str_('ke'), 19: np.str_('iniend'), 20: np.str_('apakah'), 21: np.str_('ingin'), 22: np.str_('untuk'), 23: np.str_('tahu'), 24: np.str_('kau'), 25: np.str_('ituend'), 26: np.str_('kita'), 27: np.str_('adalah'), 28: np.str_('kami'), 29: np.str_('sudah'), 30: np.str_('suka'), 31: np.str_('tomend'), 32: np.str_('pergi'), 33: np.str_('harus'), 34: np.str_('dan'), 35: np.str_('sedang'), 36: np.str_('punya'), 37: np.str_('lebih'), 38: np.str_('dari'), 39: np.str_('orang'), 40: np.str_('sangat'), 41: np.str_('anda'), 42: np.str_('mereka'), 43: np.str_('mary'), 44: np.str_('makan'), 45: np.str_('dalam'), 46: np.str_('pada'), 47: np.str_('pernah'), 48:

In [27]:
shuffled = dataset.shuffle(VOCAB_SIZE, seed=8)
n_val = int(VOCAB_SIZE * 0.2)
val_raw, train_raw = shuffled.take(n_val), shuffled.skip(n_val)

train_dataset = (train_raw.map(vectorize).shuffle(4096).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))
val_dataset = (val_raw.map(vectorize).shuffle(4096).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))

# **Encoder**

In [29]:
class Encoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, hidden_units):
        super(Encoder, self).__init__()
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
        self.LSTM = tf.keras.layers.LSTM(hidden_units, return_sequences=True, return_state=True)

    def call(self, x):
        x = self.embedding(x) # kasih padding
        return self.LSTM(x) # output terus masuk lagi

# **Implementasi Encoder**

In [30]:
# Implementasi Encoder
input_layer = tf.keras.layers.Input(shape=[ENG_SEQ_LENGTH], )
encoder = Encoder(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_UNITS)
encoder_output = encoder(input_layer)

# **Test**

In [31]:
# Test
sentences = tf.constant(["Hello!", "I am a Student."])
vectorized = eng_vec_layer(sentences)
result = encoder(vectorized)
print(result)

(<tf.Tensor: shape=(2, 16, 256), dtype=float32, numpy=
array([[[ 2.3996837e-03,  1.0164049e-03,  3.2849275e-03, ...,
          4.9036774e-03, -5.9973532e-03,  2.8200841e-03],
        [ 3.4430944e-03,  2.8206937e-03,  8.5466905e-03, ...,
          3.2984074e-03, -7.4820342e-03, -2.6495266e-04],
        [ 3.4791355e-03,  4.2714803e-03,  1.1707848e-02, ...,
          1.8431239e-03, -7.9487562e-03, -3.4580829e-03],
        ...,
        [-3.9197523e-03,  1.0787861e-02,  9.0578878e-03, ...,
         -5.1681087e-03, -4.3736971e-03, -2.2283003e-02],
        [-4.2395452e-03,  1.0918023e-02,  8.6479234e-03, ...,
         -5.2886009e-03, -4.2360565e-03, -2.2685632e-02],
        [-4.5084287e-03,  1.1015851e-02,  8.3063878e-03, ...,
         -5.3795506e-03, -4.1254717e-03, -2.2992881e-02]],

       [[ 1.9804281e-03, -3.6158541e-03, -1.2775837e-03, ...,
         -9.6555068e-06,  2.1990100e-03,  6.7536845e-03],
        [-1.9922720e-03, -1.9191179e-03, -3.3708657e-03, ...,
          6.8816640e-03,  1.